# Laboratorio en clase: SARSA con CliffWalking

**Duración:** 60–90 minutos  
**Propósito:** experimentar con políticas de selección de acciones sin tener que construir todo el algoritmo desde cero.

Este laboratorio **no es un reto evaluable**. Puedes cambiar parámetros y políticas, ejecutar nuevamente las celdas y discutir lo que observas.

## Al terminar podrás

1. Identificar la secuencia `S, A, R, S', A'` usada por SARSA.
2. Explicar qué significa que SARSA sea *on-policy*.
3. Modificar una política aleatoria, greedy, ε-greedy o softmax.
4. Comparar recompensa, duración y caídas al precipicio.
5. Interpretar una política tabular representada con flechas.

> Ejecuta las celdas en orden. Solo debes editar las que tienen el título **🧪 ACTIVIDAD**.

## 1. Preparación

La siguiente celda importa las herramientas y concentra los parámetros que modificaremos. Si aparece un error de importación, verifica que abriste Jupyter desde la raíz del repositorio.

In [3]:
import gymnasium as gym
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from labs.lab_sarsa_cliffwalking.sarsa_lab import (
    epsilon_greedy_policy,
    greedy_policy,
    moving_average,
    policy_grid,
    random_policy,
    rollout_policy,
    softmax_policy,
    train_sarsa,
)

CONFIG = {
    "episodes": 600,
    "alpha": 0.5,
    "gamma": 1.0,
    "epsilon": 0.1,
    "seed": 2026,
}

CONFIG

ModuleNotFoundError: No module named 'gymnasium'

## 2. Conocer la interfaz de Gymnasium

`CliffWalking-v1` representa una cuadrícula de 4 × 12:

- `S`: inicio.
- `G`: meta.
- `C`: precipicio.
- Cada movimiento normal entrega recompensa `-1`.
- Caer al precipicio entrega `-100` y devuelve el agente al inicio.

Las acciones son `0 = ↑`, `1 = →`, `2 = ↓`, `3 = ←`.

Gymnasium representa cada casilla con un número:

```text
 0   1   2   3   4   5   6   7   8   9  10  11
12  13  14  15  16  17  18  19  20  21  22  23
24  25  26  27  28  29  30  31  32  33  34  35
 S   C   C   C   C   C   C   C   C   C   C   G
36  37  38  39  40  41  42  43  44  45  46  47
```

El inicio es `S = 36`, la meta es `G = 47` y los estados `37` a `46` forman el precipicio. Desde `S = 36`, subir (`0`) lleva al estado `24`. Avanzar a la derecha (`1`) cae inmediatamente al precipicio y el entorno devuelve al agente a `36`.

La interfaz siempre sigue esta idea:

```python
state, info = env.reset()
next_state, reward, terminated, truncated, info = env.step(action)
```

Aquí `state` es $S$, `action` es $A$, `reward` es $R$ y `next_state` es $S'$. Todavía falta elegir $A'$; esa elección es la que completa SARSA.

In [ ]:
env_demo = gym.make("CliffWalking-v1", render_mode="ansi")
state, info = env_demo.reset(seed=CONFIG["seed"])

print(env_demo.render())
print("Estado inicial:", state)
print("Número de estados:", env_demo.observation_space.n)
print("Número de acciones:", env_demo.action_space.n)

In [ ]:
# Mapa visual: cada número es el estado que Gymnasium entrega al agente.
fig, ax = plt.subplots(figsize=(14, 4))
background = np.zeros((4, 12))
background[3, 1:11] = -1  # precipicio
background[3, 11] = 1     # meta
ax.imshow(background, cmap="RdYlGn", vmin=-1, vmax=1)

for state_number in range(48):
    row, column = divmod(state_number, 12)
    marker = ""
    if state_number == 36:
        marker = "\nS"
    elif 37 <= state_number <= 46:
        marker = "\nC"
    elif state_number == 47:
        marker = "\nG"
    ax.text(column, row, f"{state_number}{marker}", ha="center", va="center", fontweight="bold")

ax.set_xticks(range(12), labels=[f"col {column}" for column in range(12)])
ax.set_yticks(range(4), labels=[f"fila {row}" for row in range(4)])
ax.set_title("Mapa numerado: S=inicio, C=precipicio, G=meta")
ax.set_xticks(np.arange(-0.5, 12, 1), minor=True)
ax.set_yticks(np.arange(-0.5, 4, 1), minor=True)
ax.grid(which="minor", color="white", linewidth=2)
plt.show()

### 🧪 ACTIVIDAD 1 — Ejecutar acciones manualmente

Haz dos intentos:

1. Usa `actions = [1]`. Predice el estado siguiente y la recompensa.
2. Diseña una secuencia que evite el precipicio: primero debe subir, luego avanzar y finalmente bajar cerca de la meta.

Antes de ejecutar, responde: **¿en qué acción esperas recibir `-100` y por qué el estado vuelve a 36?**

> Pista: llegar a la meta desde el inicio sin caer requiere al menos 13 movimientos.

In [ ]:
# Modifica esta lista. Acciones: 0=↑, 1=→, 2=↓, 3=←
actions = [1, 0, 1, 1]

state, _ = env_demo.reset(seed=CONFIG["seed"])
for action in actions:
    next_state, reward, terminated, truncated, _ = env_demo.step(action)
    print(f"s={state:2d}, a={action}, r={reward:4.0f}, s'={next_state:2d}")
    state = next_state
    if terminated or truncated:
        break

print(env_demo.render())

## 3. La actualización SARSA

El nombre **SARSA** resume la experiencia usada en una actualización:

```text
S  →  A  →  R  →  S'  →  A'
estado  acción  recompensa  estado siguiente  acción siguiente
```

SARSA actualiza el valor de la pareja estado–acción usando la **siguiente acción que la misma política seleccionó**:

$$Q(S,A) \leftarrow Q(S,A) + \alpha [R + \gamma Q(S',A') - Q(S,A)]$$

| Símbolo | Significado | Pregunta intuitiva |
|---|---|---|
| $Q(S,A)$ | valor actual de hacer $A$ en $S$ | ¿Qué tan buena creíamos que era esta decisión? |
| $R$ | recompensa observada | ¿Qué ocurrió inmediatamente? |
| $Q(S',A')$ | valor estimado de la próxima decisión real | ¿Qué esperamos después? |
| $\alpha$ | tasa de aprendizaje | ¿Cuánto corregimos nuestra estimación? |
| $\gamma$ | factor de descuento | ¿Cuánto importa el futuro? |

El término entre corchetes es el **error TD**: la diferencia entre lo que ocurrió más lo que esperamos y lo que creíamos antes.

### El ciclo completo

```text
Inicializar Q
Para cada episodio:
    observar S
    elegir A usando la política (por ejemplo, ε-greedy)
    repetir:
        ejecutar A; observar R y S'
        si S' es terminal:
            actualizar usando solamente R y terminar el episodio
        elegir A' usando LA MISMA política
        actualizar Q(S,A) usando R + γQ(S',A')
        S ← S'
        A ← A'
```

Se llama **on-policy** porque aprende sobre la política que realmente está usando, incluida su exploración.

In [ ]:
# Ejemplo numérico: seguimos cada término de una actualización.
def mostrar_actualizacion_sarsa(q, state, action, reward, next_state, next_action, alpha, gamma):
    valor_anterior = q[state, action]
    valor_siguiente = q[next_state, next_action]
    td_target = reward + gamma * valor_siguiente
    td_error = td_target - valor_anterior
    q[state, action] += alpha * td_error
    return valor_anterior, valor_siguiente, td_target, td_error, q[state, action]

q_example = np.zeros((48, 4))
q_example[36, 0] = -2.0   # Antes creíamos que subir desde S valía -2.
q_example[24, 1] = -4.0   # La siguiente acción elegida tiene valor -4.

anterior, siguiente, target, error, nuevo = mostrar_actualizacion_sarsa(
    q_example, state=36, action=0, reward=-1,
    next_state=24, next_action=1, alpha=0.5, gamma=1.0,
)

print(f"1. Valor anterior:       Q(S,A)       = {anterior:.1f}")
print(f"2. Valor siguiente:      Q(S',A')     = {siguiente:.1f}")
print(f"3. Objetivo TD:          R + γQ(S',A') = -1 + 1×({siguiente:.1f}) = {target:.1f}")
print(f"4. Error TD:             target-Q(S,A) = {target:.1f}-({anterior:.1f}) = {error:.1f}")
print(f"5. Nuevo valor:          {anterior:.1f} + 0.5×({error:.1f}) = {nuevo:.1f}")

assert np.isclose(nuevo, -3.5)

### Comprobación rápida

Sin ejecutar código, responde:

1. Si el objetivo TD es **mejor** que el valor anterior, ¿Q aumenta o disminuye?
2. ¿Qué ocurriría con la actualización si `alpha = 0`?
3. ¿Qué ocurriría con el futuro estimado si `gamma = 0`?

<details>
<summary>Ver respuestas</summary>

1. Q aumenta; el error TD es positivo.
2. Q no cambiaría, porque el agente no incorporaría información nueva.
3. El objetivo sería solo la recompensa inmediata $R$.

</details>

### ¿Por qué importa $A'$? SARSA frente a Q-learning

La diferencia esencial está en el objetivo de actualización:

| Algoritmo | Objetivo TD | Pregunta que responde |
|---|---|---|
| SARSA | $R + \gamma Q(S',A')$ | ¿Qué valor tiene la acción que mi política realmente eligió? |
| Q-learning | $R + \gamma \max_a Q(S',a)$ | ¿Qué valor tiene la mejor acción posible? |

Con ε-greedy, SARSA sabe que a veces explorará y podría acercarse al precipicio. Por ello puede preferir una ruta más larga pero segura. Q-learning aprende suponiendo una elección greedy futura, aunque durante el entrenamiento esté explorando.

> No significa que SARSA siempre sea “mejor”: aprende una política coherente con su comportamiento de exploración.

## 4. Entrenar SARSA con una política ε-greedy

Con probabilidad `epsilon`, la política explora con una acción aleatoria. En caso contrario, elige una acción con valor Q máximo. La semilla hace que el experimento sea reproducible.

In [ ]:
env_train = gym.make("CliffWalking-v1")
result = train_sarsa(
    env_train,
    policy=epsilon_greedy_policy,
    episodes=CONFIG["episodes"],
    alpha=CONFIG["alpha"],
    gamma=CONFIG["gamma"],
    epsilon=CONFIG["epsilon"],
    seed=CONFIG["seed"],
)

print("Forma de la tabla Q:", result.q_table.shape)
print("Recompensa media, últimos 50 episodios:", result.rewards[-50:].mean())
print("Caídas, últimos 50 episodios:", result.cliff_falls[-50:].sum())

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
episodes = np.arange(1, CONFIG["episodes"] + 1)

axes[0].plot(episodes, result.rewards, alpha=0.25, label="Por episodio")
axes[0].plot(episodes, moving_average(result.rewards, 30), label="Promedio móvil")
axes[0].set(title="Recompensa", xlabel="Episodio", ylabel="Retorno")
axes[0].legend()

axes[1].plot(episodes, moving_average(result.episode_lengths.astype(float), 30))
axes[1].set(title="Duración", xlabel="Episodio", ylabel="Pasos")

axes[2].plot(episodes, moving_average(result.cliff_falls.astype(float), 30))
axes[2].set(title="Caídas", xlabel="Episodio", ylabel="Caídas")

plt.tight_layout()
plt.show()

### Cómo leer estas gráficas

No busques una línea perfectamente ascendente: el agente sigue explorando y los episodios individuales pueden variar.

- **Recompensa:** valores menos negativos son mejores. Caer cuesta `-100`, así que afecta mucho el retorno.
- **Duración:** menos pasos suele indicar una ruta más corta, pero una ruta corta junto al precipicio puede ser arriesgada.
- **Caídas:** deberían hacerse menos frecuentes a medida que mejora la política, aunque ε-greedy puede seguir cayendo por exploración.
- **Promedio móvil:** suaviza el ruido y permite observar la tendencia; no es otra recompensa distinta.

**Pausa y explica:** ¿qué evidencia muestra aprendizaje? Usa al menos dos de las tres métricas.

## 5. Leer y ejecutar la política aprendida

Durante el **entrenamiento**, ε-greedy explora: algunas veces elige una acción aleatoria. En la **evaluación** siguiente usamos una política completamente greedy (`argmax`), sin exploración, para observar qué aprendió la tabla Q.

| Fase | ¿Explora? | Propósito |
|---|---:|---|
| Entrenamiento | Sí, según ε | Recoger experiencia y actualizar Q |
| Evaluación | No | Medir la política aprendida |

Las flechas muestran la acción greedy con mayor valor en cada estado. `S`, `C` y `G` sustituyen las flechas en la fila inferior.

> Una evaluación exitosa no significa que todos los episodios de entrenamiento fueron exitosos: son fases distintas.

In [ ]:
ACTION_ARROWS = np.array(["↑", "→", "↓", "←"])


def visualize_policy_route(q_table, title, seed=2026):
    """Muestra valores, acciones greedy y recorrido final en dos paneles."""
    evaluation_env = gym.make("CliffWalking-v1")
    states, total_reward = rollout_policy(
        evaluation_env, q_table, seed=seed, max_steps=200
    )
    reached_goal = states[-1] == 47
    steps = len(states) - 1

    fig, axes = plt.subplots(1, 2, figsize=(16, 4.8))

    # Panel 1: valor del mejor movimiento y acción greedy por estado.
    state_values = np.max(q_table, axis=1).reshape(4, 12)
    image = axes[0].imshow(state_values, cmap="viridis")
    greedy_actions = np.argmax(q_table, axis=1).reshape(4, 12)
    for row in range(4):
        for column in range(12):
            state_number = row * 12 + column
            if state_number == 36:
                label = "S"
            elif 37 <= state_number <= 46:
                label = "C"
            elif state_number == 47:
                label = "G"
            else:
                label = ACTION_ARROWS[greedy_actions[row, column]]
            axes[0].text(column, row, label, ha="center", va="center", color="white", fontweight="bold")
    axes[0].set(title="Valor aprendido y acción greedy", xticks=range(12), yticks=range(4))
    fig.colorbar(image, ax=axes[0], label="max Q(s,a)", shrink=0.8)

    # Panel 2: terreno y orden del recorrido de evaluación.
    terrain = np.zeros((4, 12))
    terrain[3, 1:11] = -1
    terrain[3, 11] = 1
    axes[1].imshow(terrain, cmap="RdYlGn", vmin=-1, vmax=1)
    coordinates = [divmod(state, 12) for state in states]
    rows = [position[0] for position in coordinates]
    columns = [position[1] for position in coordinates]
    axes[1].plot(columns, rows, color="#1565c0", linewidth=3, marker="o", markersize=5)

    first_visit = {}
    for step_number, state_number in enumerate(states):
        first_visit.setdefault(state_number, step_number)
    if len(first_visit) <= 30:
        for state_number, step_number in first_visit.items():
            row, column = divmod(state_number, 12)
            axes[1].text(column, row, step_number, ha="center", va="center", fontsize=8)

    axes[1].scatter([0], [3], color="red", marker="s", s=130, label="Inicio")
    end_marker = "*" if reached_goal else "X"
    end_color = "green" if reached_goal else "orange"
    axes[1].scatter([columns[-1]], [rows[-1]], color=end_color, marker=end_marker, s=180, label="Final")
    axes[1].set(title="Recorrido greedy final (números = paso)", xticks=range(12), yticks=range(4))
    axes[1].legend(loc="upper right")

    status = "llegó a la meta" if reached_goal else "no llegó en 200 pasos"
    fig.suptitle(
        f"{title} — {status} | pasos={steps} | recompensa={total_reward:.0f}",
        fontsize=14,
        fontweight="bold",
    )
    plt.tight_layout()
    plt.show()
    return states, total_reward


visited_states, evaluation_reward = visualize_policy_route(
    result.q_table, "SARSA con ε-greedy", seed=CONFIG["seed"]
)

In [ ]:
grid = policy_grid(result.q_table)
print("Política en formato de texto:")
for row in grid:
    print("  ".join(row))
print("\nEstados visitados:", visited_states)
print("Recompensa de evaluación:", evaluation_reward)
print("¿Llegó a la meta?", visited_states[-1] == 47)

## 6. 🧪 ACTIVIDAD 2 — Cambiar la política

Cambia únicamente `POLICY_NAME` y ejecuta las siguientes dos celdas. Opciones:

- `random`: todas las acciones tienen la misma probabilidad.
- `greedy`: siempre elige una acción de máximo valor.
- `epsilon_greedy`: explora con probabilidad ε.
- `softmax`: todas las acciones pueden elegirse según sus valores Q.

Antes de ejecutar, escribe una predicción: **¿cuál política tendrá menos caídas y cuál aprenderá más rápido?**

In [ ]:
# MODIFICA ESTA LÍNEA:
POLICY_NAME = "epsilon_greedy"

POLICIES = {
    "random": (random_policy, {}),
    "greedy": (greedy_policy, {}),
    "epsilon_greedy": (epsilon_greedy_policy, {"epsilon": 0.1}),
    "softmax": (softmax_policy, {"temperature": 0.5}),
}

selected_policy, policy_parameters = POLICIES[POLICY_NAME]
print("Política seleccionada:", POLICY_NAME)
print("Parámetros:", policy_parameters)

In [ ]:
selected_result = train_sarsa(
    gym.make("CliffWalking-v1"),
    policy=selected_policy,
    episodes=CONFIG["episodes"],
    alpha=CONFIG["alpha"],
    gamma=CONFIG["gamma"],
    seed=CONFIG["seed"],
    **policy_parameters,
)

print("Recompensa media final:", selected_result.rewards[-50:].mean())
print("Pasos medios finales:", selected_result.episode_lengths[-50:].mean())
print("Caídas finales:", selected_result.cliff_falls[-50:].sum())

selected_states, selected_reward = visualize_policy_route(
    selected_result.q_table,
    f"Política seleccionada: {POLICY_NAME}",
    seed=CONFIG["seed"],
)

## 7. 🧪 ACTIVIDAD 3 — Construir y modificar ε-greedy

Esta actividad tiene dos niveles. Primero completas las decisiones esenciales de la política; después modificas una versión funcional para experimentar.

$$\pi(a|s)=\begin{cases}
\text{acción aleatoria}, & \text{con probabilidad } \epsilon\\
\text{una acción de valor máximo}, & \text{con probabilidad } 1-\epsilon
\end{cases}$$

### Nivel 1 — Construcción guiada

1. `rng.random()` produce un número entre 0 y 1.
2. Si es menor que `epsilon`, se devuelve una acción aleatoria.
3. En caso contrario, se encuentran las acciones con valor máximo.
4. Si hay empate, se elige aleatoriamente entre las mejores para no favorecer siempre la acción 0.

Antes de entrenar, predice:

- Con `epsilon = 0`, ¿habrá exploración explícita?
- Con `epsilon = 1`, ¿se usarán los valores Q para escoger acciones?
- ¿Por qué resolver empates importa cuando Q comienza llena de ceros?

Completa los tres espacios de la celda siguiente reemplazando `None`. No escribas una función completa dentro del espacio: cada `None` se reemplaza por **una sola expresión**.

| Espacio | Debe producir | Herramientas permitidas |
|---|---|---|
| TODO 1 | Un booleano: `True` si corresponde explorar | `rng.random()`, `<`, `epsilon` |
| TODO 2 | Un entero entre `0` y `len(q_values)-1` | `rng.integers`, `len`, `int` |
| TODO 3 | Un arreglo con los índices empatados en el máximo | `np.flatnonzero`, comparación, `np.max` |

La comprobación te dirá qué regla falta sin detener la ejecución completa del notebook.

In [ ]:
# Reemplaza los tres None. No cambies el resto de la función.
def guided_epsilon_greedy(q_values, rng, epsilon=0.1, **kwargs):
    explore = None  # TODO 1: expresión booleana; ejemplo de tipo esperado: bool
    if explore:
        return None  # TODO 2: expresión que devuelve int en [0, número de acciones)

    best_actions = None  # TODO 3: arreglo 1D de índices; conserva todos los empates
    if best_actions is None:
        return None
    return int(rng.choice(best_actions))


# Comprobación formativa: informa el avance sin bloquear Run All.
probe_q = np.array([0.0, 3.0, 1.0, 3.0])
probe_rng = np.random.default_rng(7)
guided_action = guided_epsilon_greedy(probe_q, probe_rng, epsilon=0.0)

if guided_action is None:
    print("Actividad pendiente: reemplaza los tres None y ejecuta de nuevo.")
elif guided_action not in (1, 3):
    print("Revisa TODO 3: con epsilon=0 debes elegir una acción greedy (1 o 3).")
else:
    print("Nivel 1 superado: la política greedy resolvió correctamente el empate.")

<details>
<summary>Pista y solución de referencia del nivel 1</summary>

```python
explore = rng.random() < epsilon
return int(rng.integers(len(q_values)))
best_actions = np.flatnonzero(q_values == np.max(q_values))
```

Usa la pista solo después de intentar relacionar cada línea con la definición matemática.
</details>

### Nivel 2 — Exploración libre

La siguiente versión ya funciona. Cambia `EXPERIMENT_EPSILON` entre `0.0`, `0.1`, `0.3` y `1.0`; predice el resultado antes de entrenar y compara recompensa y caídas.

In [ ]:
def experimental_epsilon_greedy(q_values, rng, epsilon=0.1, **kwargs):
    """Versión funcional que puedes modificar libremente."""
    if rng.random() < epsilon:
        return int(rng.integers(len(q_values)))
    best_actions = np.flatnonzero(q_values == np.max(q_values))
    return int(rng.choice(best_actions))


EXPERIMENT_EPSILON = 0.1  # Prueba 0.0, 0.1, 0.3 y 1.0
my_result = train_sarsa(
    gym.make("CliffWalking-v1"),
    policy=experimental_epsilon_greedy,
    episodes=CONFIG["episodes"],
    alpha=CONFIG["alpha"],
    gamma=CONFIG["gamma"],
    epsilon=EXPERIMENT_EPSILON,
    seed=CONFIG["seed"],
)

print("Nivel 2 ejecutado con epsilon =", EXPERIMENT_EPSILON)
print("Recompensa media final:", my_result.rewards[-50:].mean())
print("Caídas en los últimos 50 episodios:", my_result.cliff_falls[-50:].sum())

my_states, my_evaluation_reward = visualize_policy_route(
    my_result.q_table,
    f"Mi ε-greedy (epsilon={EXPERIMENT_EPSILON})",
    seed=CONFIG["seed"],
)

## 8. Comparación justa

Ejecutaremos todas las políticas con los mismos episodios, hiperparámetros y semilla. Cambiar varias condiciones a la vez impediría saber qué causó la diferencia.

Antes de mirar la tabla, recuerda:

- Una recompensa menos negativa es mejor (`-20` es mejor que `-100`).
- Menos pasos no siempre significa una política más segura.
- Una sola semilla permite repetir exactamente el experimento, pero no demuestra que el resultado sea general.
- La política aleatoria no mejora su comportamiento aunque la tabla Q sí se actualice: la política sigue ignorando Q al actuar.

In [ ]:
comparison_rows = []
comparison_results = {}

for name, (policy, parameters) in POLICIES.items():
    current = train_sarsa(
        gym.make("CliffWalking-v1"),
        policy=policy,
        episodes=CONFIG["episodes"],
        alpha=CONFIG["alpha"],
        gamma=CONFIG["gamma"],
        seed=CONFIG["seed"],
        **parameters,
    )
    comparison_results[name] = current
    comparison_rows.append({
        "política": name,
        "recompensa_final": current.rewards[-50:].mean(),
        "pasos_finales": current.episode_lengths[-50:].mean(),
        "caídas_finales": current.cliff_falls[-50:].sum(),
    })

comparison = pd.DataFrame(comparison_rows).sort_values(
    "recompensa_final", ascending=False
)
comparison

In [ ]:
plt.figure(figsize=(10, 5))
for name, current in comparison_results.items():
    plt.plot(
        moving_average(current.rewards, 30),
        label=name,
    )
plt.xlabel("Episodio")
plt.ylabel("Recompensa media móvil")
plt.title("SARSA con diferentes políticas de comportamiento")
plt.legend()
plt.show()

### Comparar también las rutas finales

Las curvas resumen el entrenamiento, pero no muestran la conducta espacial. Ejecuta la siguiente celda para ver qué ruta greedy dejó cada política después de entrenar bajo las mismas condiciones.

> Si una política no llega a la meta en 200 pasos, la figura lo indicará. Eso también es un resultado: significa que su tabla Q no produjo una política greedy final exitosa con esta configuración.

In [ ]:
comparison_routes = {}
for name, current in comparison_results.items():
    states, reward = visualize_policy_route(
        current.q_table,
        f"Comparación final: {name}",
        seed=CONFIG["seed"],
    )
    comparison_routes[name] = {
        "llegó_meta": states[-1] == 47,
        "pasos": len(states) - 1,
        "recompensa_evaluación": reward,
    }

pd.DataFrame(comparison_routes).T

## 9. 🧪 ACTIVIDAD 4 — Experimento y conclusión

Elige **una** pregunta:

1. ¿Cómo cambia el aprendizaje con `epsilon = 0.0, 0.1, 0.3, 1.0`?
2. ¿Cómo cambia softmax con `temperature = 0.1, 0.5, 1.0, 5.0`?
3. ¿Qué ocurre al cambiar `alpha` entre `0.1`, `0.5` y `1.0`?

Cambia un solo factor, conserva la semilla y registra los resultados. Después responde:

- ¿Qué modificaste?
- ¿Qué esperabas que ocurriera?
- ¿Qué muestran la recompensa y las caídas?
- ¿La evidencia apoya tu predicción?
- ¿Qué limitación tiene comparar una sola semilla?

### Plantilla de conclusión

> Al cambiar ______ de ______ a ______, observamos que ______. La recompensa media cambió de ______ a ______ y las caídas cambiaron de ______ a ______. Esto sugiere que ______. Sin embargo, para obtener una conclusión más sólida deberíamos ______.

## Lista de verificación y explicación final

Antes de terminar, confirma:

- [ ] Ejecuté acciones manuales y observé una caída.
- [ ] Puedo identificar `S, A, R, S', A'` en una transición.
- [ ] Puedo explicar con palabras qué representa el error TD.
- [ ] Puedo explicar por qué SARSA es *on-policy*.
- [ ] Distingo entrenamiento con exploración de evaluación greedy.
- [ ] Interpreté la cuadrícula de flechas y las tres métricas.
- [ ] Cambié al menos dos políticas.
- [ ] Modifiqué ε y expliqué su efecto.
- [ ] Comparé condiciones cambiando un solo factor.
- [ ] Escribí una conclusión basada en métricas y mencioné la limitación de usar una semilla.

### Explícalo sin fórmulas

Completa estas frases con tus propias palabras:

1. La tabla Q representa ______________________________________________.
2. SARSA usa la siguiente acción $A'$ porque ____________________________.
3. Aumentar ε cambia el comportamiento porque __________________________.
4. Una ruta más larga puede ser preferible cuando ______________________.

### Idea central

SARSA aprende de lo que su política **realmente hace**, no de lo que haría una política ideal. La siguiente acción $A'$ afecta directamente la actualización; por eso la exploración cambia tanto las experiencias recogidas como la política aprendida.